# TP final IA II - Paso 5: exportar los datos y modelos para la página web

Este notebook **no entrena nada**. Toma lo que ya generaron el 02, el 03 y el 04 y arma la carpeta `datos_app/` que usa la página (mapa, inspector de lotes y API `/predict`):

| Archivo | Qué tiene |
|---|---|
| `lotes.csv` | Los 18.692 lotes: ubicación, campaña, departamento, clase del INTA, predicción de la red **en enero** y **con la campaña completa** (modelo que no vio esa campaña), probabilidades, aciertos y rindes |
| `curvas_ndvi.npz` | La curva de NDVI de cada lote (27 períodos de 10 días) |
| `curvas_tipicas.csv` | La curva típica de cada cultivo por campaña y departamento (y de toda la zona), para comparar un lote con su zona |
| `metricas.json` | F1 por campaña, matriz "según el INTA → según la red" por campaña, MAE de rinde |
| `modelos/` | Los 12 modelos de año afuera (6 de campaña completa y 6 hasta enero) **en float32**, cada uno con su normalización |
| `ejemplos/` | Unos 200 lotes con su serie **cruda** de Sentinel-2 (10 bandas, uint16) para que `/predict` corra el modelo de verdad, y archivos de ejemplo para descargar desde la página |
| `inferencia.py` | El código de referencia para correr los modelos (preprocesamiento igual al del 03). La API lo importa tal cual |
| `versiones.json` | Versiones de TensorFlow, Keras y NumPy con las que se exportó |

Al final se verifica que **todos los modelos cargan y predicen en CPU** y que dan lo mismo que en el 03 y el 04 (diferencia máxima de probabilidad menor a 0,01 y la misma clase en al menos el 99,5 % de los lotes).

**Antes de correrlo:** el 04 tiene que haber terminado con los modelos "hasta enero" guardados. Entorno **CPU** (no hace falta GPU). Tarda unos 15 a 25 minutos.

In [2]:
PRUEBA = False   # True solo para probar el circuito con los resultados *_prueba

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/TP_final_IA2'
except ImportError:
    BASE = os.environ.get('TP_BASE', 'TP_final_IA2')

DATASET = f'{BASE}/dataset_v2'
R03 = f'{BASE}/resultados_03v2_prueba' if PRUEBA else f'{BASE}/resultados_03v2'
R04 = f'{BASE}/resultados_04_prueba' if PRUEBA else f'{BASE}/resultados_04'
LOCAL = '/content/datos_app' if os.path.exists('/content') else os.path.abspath('datos_app')   # se arma local y al final se copia a Drive
DESTINO = f'{BASE}/datos_app_prueba' if PRUEBA else f'{BASE}/datos_app'
for ruta in [f'{DATASET}/dataset_cultivos.npz', f'{DATASET}/dataset_meta.csv', f'{R03}/particiones_03.csv']:
    assert os.path.exists(ruta), f'Falta {ruta}'
print('03:', R03, '\n04:', R04, '\nSalida:', DESTINO)

Mounted at /content/drive
03: /content/drive/MyDrive/TP_final_IA2/resultados_03v2 
04: /content/drive/MyDrive/TP_final_IA2/resultados_04 
Salida: /content/drive/MyDrive/TP_final_IA2/datos_app


In [3]:
import json, shutil, time, glob
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score

SEED = 42
CLASES = ['Maíz', 'Soja 1ra', 'Soja 2da', 'Otros']
BANDAS_S2 = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
CANALES = BANDAS_S2 + ['NDVI', 'NDRE']
T_FULL, T_ENE, PATCH, N_PRUEBA = 27, 15, 16, 600
OBJ = 'rinde_sembrado'
N_EJEMPLOS_POR_GRUPO = 8          # por campaña y clase: 6 x 4 x 8 = 192 lotes con su serie cruda
I_NDVI = CANALES.index('NDVI')

shutil.rmtree(LOCAL, ignore_errors=True)
for sub in ['modelos', 'ejemplos', 'ejemplos/descargas']:
    os.makedirs(f'{LOCAL}/{sub}', exist_ok=True)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'no (está bien)')

TensorFlow 2.21.0 | Keras 3.13.2 | GPU: no (está bien)


## 1. El código de inferencia (`inferencia.py`)

Es el **único lugar** donde se define cómo se prepara una serie para el modelo. La API lo usa tal cual, así nadie calcula los índices o la normalización de otra forma. Recibe la serie **cruda** de Sentinel-2: un arreglo `(27, 16, 16, 10)` en `uint16`, con las 10 bandas en el orden de `BANDAS_S2`, un período cada 10 días del 1 de septiembre al 31 de mayo.

> **Actualizado después de ejecutar:** se agregó `validar_serie()`, que rechaza series con píxeles sin dato o que no vienen en `uint16` (ver la revisión en el README). Es el mismo archivo que `app/inferencia.py`. No cambia ninguna salida de este notebook: los 192 lotes de ejemplo pasan la validación.


In [4]:
INFERENCIA = r"""
# Inferencia de los modelos del TP final de IA II (UCA Rosario, 2026).
#
# Uso:
#     from inferencia import Modelos
#     m = Modelos('datos_app')                      # carga los 12 modelos (tarda unos segundos)
#     r = m.predecir(bandas, campana='2022_23')     # bandas: (27, 16, 16, 10) uint16
#     r = m.predecir(bandas)                        # sin campaña: promedio de los 6 modelos
#
# Ojo con el modo sin campaña: si el lote es de una de las 6 campañas del trabajo, 5 de esos 6 modelos lo vieron
# al entrenar y el resultado es optimista. Para evaluar un lote del dataset, pasar su campaña.
#
# La serie tiene que venir preparada como en el notebook 02: nubes enmascaradas, mediana cada 10 días y huecos
# interpolados. validar_serie() rechaza series con huecos (píxeles en 0), que el modelo respondería sin sentido.
import json, os
import numpy as np
import keras
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

CLASES = ['Maíz', 'Soja 1ra', 'Soja 2da', 'Otros']
BANDAS_S2 = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
T_FULL, T_ENE, PATCH = 27, 15, 16


class SerieInvalida(ValueError):
    pass


def validar_serie(bandas, max_ceros=0.01):
    # Rechaza series que no vienen preparadas como en el notebook 02. En los 192 lotes reales de ejemplo no hay
    # ningún píxel-período en 0 en todas las bandas, y como mucho un 0,07 % tiene un 0 en alguna banda.
    x = np.asarray(bandas)
    if x.ndim == 4:
        x = x[None]
    if x.shape[1:] != (T_FULL, PATCH, PATCH, len(BANDAS_S2)):
        raise SerieInvalida(f'Se esperaba (27, 16, 16, 10) por lote y llegó {x.shape[1:]}')
    if x.dtype != np.uint16:
        raise SerieInvalida(f'Se esperaban las bandas crudas en uint16 (valores de 0 a 65535) y llegó {x.dtype}')
    huecos = (x == 0).all(-1)                       # píxel-período sin dato en ninguna banda
    if huecos.any():
        n, t = int(huecos.sum()), sorted(set(np.where(huecos)[1].tolist()))
        raise SerieInvalida(f'La serie tiene {n} píxeles sin dato (0 en todas las bandas) en los períodos {t}. '
                            'El modelo espera la serie preparada como en el notebook 02: nubes enmascaradas, '
                            'mediana cada 10 días y huecos interpolados.')
    frac = float((x == 0).any(-1).mean())
    if frac > max_ceros:
        raise SerieInvalida(f'El {100 * frac:.1f} % de los píxeles tiene alguna banda en 0 (se tolera hasta '
                            f'{100 * max_ceros:.0f} %). Revisar el enmascarado de nubes y la interpolación.')


def preprocesar(bandas):
    # (n, 27, 16, 16, 10) o (27, 16, 16, 10) en uint16 -> (n, 27, 16, 16, 12) float32 sin normalizar.
    # Igual que el notebook 03: reflectancia = valor / 10000, y se agregan NDVI y NDRE.
    x = np.asarray(bandas)
    if x.ndim == 4:
        x = x[None]
    if x.shape[1:] != (T_FULL, PATCH, PATCH, len(BANDAS_S2)):
        raise ValueError(f'Se esperaba (27, 16, 16, 10) por lote y llegó {x.shape[1:]}')
    x = x.astype(np.float32) / 10000.0
    b = {n: i for i, n in enumerate(BANDAS_S2)}
    ndvi = (x[..., b['B8']] - x[..., b['B4']]) / np.maximum(x[..., b['B8']] + x[..., b['B4']], 1e-6)
    ndre = (x[..., b['B8A']] - x[..., b['B5']]) / np.maximum(x[..., b['B8A']] + x[..., b['B5']], 1e-6)
    # el 03 guardaba la serie en float16: se redondea igual para reproducir exactamente lo que vio el modelo
    return np.concatenate([x, ndvi[..., None], ndre[..., None]], axis=-1).astype(np.float16).astype(np.float32)


class Modelos:
    def __init__(self, carpeta):
        self.carpeta = carpeta
        self.indice = json.load(open(os.path.join(carpeta, 'modelos', 'indice.json')))
        self.campanas = sorted(self.indice)
        self._m = {}
        for camp, modos in self.indice.items():
            for modo, info in modos.items():
                modelo = keras.models.load_model(os.path.join(carpeta, 'modelos', info['archivo']), compile=False)
                self._m[(camp, modo)] = (modelo, info)

    def _correr(self, x12, camp, modo):
        modelo, info = self._m[(camp, modo)]
        T = info['periodos']
        mu = np.asarray(info['mu_canales'], np.float32)
        sd = np.asarray(info['sd_canales'], np.float32)
        x = ((x12[:, :T] - mu) / sd).astype(np.float32)
        entrada = {'serie': x} if isinstance(modelo.input, dict) else x
        out = modelo.predict(entrada, verbose=0, batch_size=64)
        prob = np.asarray(out['cultivo'], np.float64)
        rinde = np.asarray(out['rinde'], np.float64).ravel() * info['rinde_desvio'] + info['rinde_media']
        return prob, rinde

    def predecir_lote(self, bandas, campana=None, modo='completa'):
        # Probabilidades (n, 4) y rinde (n,) en kg/ha. modo: 'completa' (27 períodos) o 'enero' (los primeros 15).
        # Con campana=None se promedian los 6 modelos (uno por campaña dejada afuera).
        validar_serie(bandas)
        x12 = preprocesar(bandas)
        camps = [campana] if campana else self.campanas
        for c in camps:
            if (c, modo) not in self._m:
                raise ValueError(f'No hay modelo para campaña={c} y modo={modo}. Campañas: {self.campanas}')
        res = [self._correr(x12, c, modo) for c in camps]
        prob = np.mean([p for p, _ in res], axis=0)
        rinde = np.mean([r for _, r in res], axis=0)
        return prob, rinde

    def predecir(self, bandas, campana=None):
        # Resultado legible para un solo lote, con el modelo en enero y con la campaña completa.
        salida = {'campana_modelo': campana or 'promedio de los 6 modelos'}
        for modo in ('enero', 'completa'):
            prob, rinde = self.predecir_lote(bandas, campana, modo)
            k = int(prob[0].argmax())
            salida[modo] = {
                'cultivo': CLASES[k],
                'probabilidades': {c: round(float(p), 4) for c, p in zip(CLASES, prob[0])},
                # el rinde solo tiene sentido para maíz y soja
                'rinde_kg_ha': None if k == 3 else round(float(rinde[0])),
            }
        return salida
"""
open(f'{LOCAL}/inferencia.py', 'w').write(INFERENCIA.lstrip())
import sys, importlib
sys.dont_write_bytecode = True
sys.path.insert(0, LOCAL)
import inferencia
importlib.reload(inferencia)
print('inferencia.py escrito')

inferencia.py escrito


## 2. Datos y particiones (igual que el 03)

In [5]:
d = np.load(f'{DATASET}/dataset_cultivos.npz')
X_u16, y_cultivo, ids = d['X'], d['y_cultivo'].astype(np.int64), d['id']
relleno = d['relleno'] if 'relleno' in d.files else None
meta = pd.read_csv(f'{DATASET}/dataset_meta.csv')
meta = meta.set_index('id').loc[ids].reset_index()
assert (meta.clase.values == y_cultivo).all()

if PRUEBA:      # la misma muestra que el 03 en modo prueba
    rng = np.random.default_rng(SEED)
    grupos_cc = meta.groupby(['campana', 'clase']).indices
    por = max(1, N_PRUEBA // len(grupos_cc))
    sel = np.sort(np.concatenate([rng.choice(v, size=min(por, len(v)), replace=False) for v in grupos_cc.values()]))
    X_u16, y_cultivo, ids, meta = X_u16[sel], y_cultivo[sel], ids[sel], meta.iloc[sel].reset_index(drop=True)
    relleno = relleno[sel] if relleno is not None else None
N = len(meta)

# Coordenadas: del meta o, si no están, de puntos.csv del 02
if not {'lon', 'lat'} <= set(meta.columns):
    ruta_p = f'{DATASET}/puntos.csv'
    if os.path.exists(ruta_p):
        p = pd.read_csv(ruta_p)
        assert {'id', 'lon', 'lat'} <= set(p.columns), 'puntos.csv no tiene id, lon y lat'
        meta = meta.merge(p[['id', 'lon', 'lat']].drop_duplicates('id'), on='id', how='left')
    elif PRUEBA:     # los datos sintéticos de prueba no tienen coordenadas
        r = np.random.default_rng(0)
        meta['lon'], meta['lat'] = r.uniform(-63.5, -59.6, N), r.uniform(-34.2, -31.5, N)
assert {'lon', 'lat'} <= set(meta.columns) and meta[['lon', 'lat']].notna().all().all(), 'Faltan coordenadas de algún lote'

# Serie de 12 canales (igual que el 03) y estadísticos para la normalización
X_all = np.empty((N, T_FULL, PATCH, PATCH, len(CANALES)), np.float16)
for i in range(0, N, 500):
    X_all[i:i + 500] = inferencia.preprocesar(X_u16[i:i + 500])
S1 = np.zeros((N, len(CANALES))); S2 = np.zeros((N, len(CANALES)))
ndvi = np.zeros((N, T_FULL), np.float32)
for i in range(0, N, 500):
    x = X_all[i:i + 500].astype(np.float32)
    S1[i:i + 500] = x.mean(axis=(1, 2, 3)); S2[i:i + 500] = (x ** 2).mean(axis=(1, 2, 3))
    ndvi[i:i + 500] = x[..., I_NDVI].mean(axis=(2, 3))

campanas = sorted(meta.campana.unique())
estrato = y_cultivo + len(CLASES) * meta.campana.map({c: i for i, c in enumerate(campanas)}).values
grupos = meta.grupo.values

def particion_interna(tr, seed):
    sg = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    a, b = min(sg.split(tr, estrato[tr], grupos[tr]), key=lambda ab: abs(len(ab[1]) / len(tr) - 0.2))
    return np.sort(tr[a]), np.sort(tr[b])

FOLDS = []
for k, camp in enumerate(campanas):
    te = np.flatnonzero(meta.campana.values == camp)
    tr = np.flatnonzero(meta.campana.values != camp)
    a, b = particion_interna(tr, SEED + 100 + k)
    FOLDS.append({'train': a, 'val': b, 'test': te, 'nombre': camp})

part = pd.read_csv(f'{R03}/particiones_03.csv').set_index('id').loc[ids]
for k, f in enumerate(FOLDS):
    assert np.array_equal(np.flatnonzero(part.test_anio.values == k), f['test']), 'Las particiones no coinciden con las del 03'

def stats_canales(idx):
    mu = S1[idx].mean(axis=0)
    sd = np.sqrt(np.maximum(S2[idx].mean(axis=0) - mu ** 2, 1e-12)) + 1e-6
    return mu.astype(np.float32), sd.astype(np.float32)

def escala_rinde(idx_train):
    yr = np.where(meta.clase.values >= 3, np.nan, meta[OBJ].values.astype(np.float64))
    return float(np.nanmean(yr[idx_train])), float(np.nanstd(yr[idx_train]))

print(f'{N} lotes | campañas: {campanas} | particiones verificadas contra el 03')

18692 lotes | campañas: ['2019_20', '2020_21', '2021_22', '2022_23', '2023_24', '2024_25'] | particiones verificadas contra el 03


## 3. Modelos: pasar a float32 y guardar con su normalización

Los modelos se entrenaron con precisión mixta (float16 en la GPU). Para el servidor, que no tiene GPU, se reconstruyen en **float32** con los mismos pesos.

In [6]:
def a_float32(modelo):
    def cambiar(o):
        if isinstance(o, dict):
            return {k: cambiar(v) for k, v in o.items()}
        if isinstance(o, list):
            return [cambiar(v) for v in o]
        return 'float32' if o == 'mixed_float16' else o
    nuevo = keras.Model.from_config(cambiar(modelo.get_config()))
    nuevo.set_weights(modelo.get_weights())
    return nuevo

def json_corrida(ruta):
    r = json.load(open(ruta))
    return np.array(r['idx_test']), np.array(r['prob'], np.float64), np.array([np.nan if v is None else v for v in r['rinde']], np.float64)

indice = {}
PRED = {}                     # (modo, k) -> (idx_test, prob, rinde) guardados por el 03 y el 04
for k, f in enumerate(FOLDS):
    camp = f['nombre']
    indice[camp] = {}
    fuentes = {
        'completa': (f'{R03}/corridas/anio/modelo_cnn_lstm_fold{k}.keras', f'{R03}/corridas/anio/cnn_lstm_fold{k}.json', None),
        'enero': (f'{R04}/corridas/anio/modelo_cnn_lstm_hasta_ene_fold{k}.keras', f'{R04}/corridas/anio/cnn_lstm_hasta_ene_fold{k}.json',
                  f'{R04}/corridas/anio/modelo_cnn_lstm_hasta_ene_fold{k}_normalizacion.json'),
    }
    for modo, (ruta_m, ruta_j, ruta_n) in fuentes.items():
        assert os.path.exists(ruta_m), f'Falta {ruta_m}' + (' (¿corrió el 04 con GUARDAR_MODELOS?)' if modo == 'enero' else '')
        assert os.path.exists(ruta_j), f'Falta {ruta_j}'
        PRED[(modo, k)] = json_corrida(ruta_j)
        if ruta_n:                       # el 04 guardó la normalización junto al modelo
            n = json.load(open(ruta_n))
            mu, sd, rm, rs = n['mu_canales'], n['sd_canales'], n['rinde_media'], n['rinde_desvio']
            assert n['campana_test'] == camp and n['periodos'] == T_ENE and not n['clima']
        else:                            # el 03 no la guardó: se recalcula con su misma fórmula
            mu, sd = [v.tolist() for v in stats_canales(f['train'])]
            rm, rs = escala_rinde(f['train'])
        m32 = a_float32(keras.models.load_model(ruta_m, compile=False))
        archivo = f'{modo}_{camp}.keras'
        m32.save(f'{LOCAL}/modelos/{archivo}')
        indice[camp][modo] = {'archivo': archivo, 'periodos': T_FULL if modo == 'completa' else T_ENE,
                              'mu_canales': list(map(float, mu)), 'sd_canales': list(map(float, sd)),
                              'rinde_media': float(rm), 'rinde_desvio': float(rs), 'campana_que_no_vio': camp}
        print(f'  {camp} {modo:8s} -> modelos/{archivo} ({m32.count_params():,} parámetros)')
json.dump(indice, open(f'{LOCAL}/modelos/indice.json', 'w'), indent=1)

  2019_20 completa -> modelos/completa_2019_20.keras (76,133 parámetros)
  2019_20 enero    -> modelos/enero_2019_20.keras (76,133 parámetros)
  2020_21 completa -> modelos/completa_2020_21.keras (76,133 parámetros)
  2020_21 enero    -> modelos/enero_2020_21.keras (76,133 parámetros)
  2021_22 completa -> modelos/completa_2021_22.keras (76,133 parámetros)
  2021_22 enero    -> modelos/enero_2021_22.keras (76,133 parámetros)
  2022_23 completa -> modelos/completa_2022_23.keras (76,133 parámetros)
  2022_23 enero    -> modelos/enero_2022_23.keras (76,133 parámetros)
  2023_24 completa -> modelos/completa_2023_24.keras (76,133 parámetros)
  2023_24 enero    -> modelos/enero_2023_24.keras (76,133 parámetros)
  2024_25 completa -> modelos/completa_2024_25.keras (76,133 parámetros)
  2024_25 enero    -> modelos/enero_2024_25.keras (76,133 parámetros)


## 4. Verificación: los modelos exportados dan lo mismo que en el 03 y el 04

Se cargan con `inferencia.py`, como lo va a hacer la API, y se comparan con las probabilidades que guardaron el 03 y el 04 para la campaña que cada modelo **no** vio. Se usa una muestra de hasta 600 lotes por modelo para que no tarde demasiado en CPU.

In [7]:
t0 = time.time()
M = inferencia.Modelos(LOCAL)
rng = np.random.default_rng(1)
filas = []
for (modo, k), (idx, prob, rinde) in PRED.items():
    camp = FOLDS[k]['nombre']
    s = np.sort(rng.choice(len(idx), size=min(600, len(idx)), replace=False))
    p, r = M.predecir_lote(X_u16[idx[s]], campana=camp, modo=modo)
    dif = np.abs(p - prob[s]).max()
    misma = (p.argmax(1) == prob[s].argmax(1)).mean()
    ok_r = np.isfinite(rinde[s])
    dif_r = np.nanmax(np.abs(r[ok_r] - rinde[s][ok_r])) if ok_r.any() else 0.0
    filas.append({'campaña': camp, 'modo': modo, 'lotes': len(s), 'dif. máx. prob.': round(float(dif), 4),
                  'misma clase (%)': round(100 * misma, 2), 'dif. máx. rinde (kg/ha)': round(float(dif_r), 1),
                  'ok': bool(dif < 0.01 and misma >= 0.995)})
ver = pd.DataFrame(filas).sort_values(['modo', 'campaña'])
print(ver.to_string(index=False))
print(f'\nVerificación en {time.time() - t0:.0f} s')
assert ver.ok.all(), 'Algún modelo exportado no reproduce las predicciones del 03/04: revisar antes de publicar'

campaña     modo  lotes  dif. máx. prob.  misma clase (%)  dif. máx. rinde (kg/ha)   ok
2019_20 completa    600           0.0000           100.00                      0.0 True
2020_21 completa    600           0.0000           100.00                      0.0 True
2021_22 completa    600           0.0000           100.00                      0.0 True
2022_23 completa    600           0.0000           100.00                      0.0 True
2023_24 completa    600           0.0000           100.00                      0.0 True
2024_25 completa    600           0.0000           100.00                      0.0 True
2019_20    enero    600           0.0007           100.00                      2.1 True
2020_21    enero    600           0.0007           100.00                      2.5 True
2021_22    enero    600           0.0011           100.00                      3.2 True
2022_23    enero    600           0.0016           100.00                      3.2 True
2023_24    enero    600         

## 5. Tabla de lotes, curvas y métricas

In [8]:
lotes = meta[['id', 'lon', 'lat', 'campana', 'provincia', 'departamento', 'grupo', 'cultivo', 'clase']].copy()
lotes = lotes.rename(columns={'cultivo': 'clase_inta', 'clase': 'clase_inta_n'})
lotes['rinde_real'] = np.where(meta.clase.values >= 3, np.nan, meta[OBJ].values)
for modo in ('enero', 'completa'):
    P = np.full((N, len(CLASES)), np.nan); R = np.full(N, np.nan)
    for k in range(len(FOLDS)):
        idx, prob, rinde = PRED[(modo, k)]
        P[idx], R[idx] = prob, rinde
    assert np.isfinite(P).all(), f'Faltan predicciones en modo {modo}'
    lotes[f'pred_{modo}'] = np.array(CLASES)[P.argmax(1)]
    for j, c in enumerate(CLASES):
        lotes[f'prob_{modo}_{c}'] = P[:, j].round(4)
    lotes[f'acierto_{modo}'] = P.argmax(1) == meta.clase.values
    lotes[f'rinde_{modo}'] = np.where(meta.clase.values >= 3, np.nan, R).round(0)

# Promedio histórico del cultivo en el departamento (otras campañas) y del cultivo en toda la zona
# (cada campaña pesa lo mismo; se calcula una vez por campaña dejando esa campaña afuera)
por_camp = meta[meta.clase < 3].groupby(['grupo', 'cultivo', 'campana'])[OBJ].first().dropna().reset_index()
hist_depto, hist_zona = np.full(N, np.nan), np.full(N, np.nan)
for camp in campanas:
    otros = por_camp[por_camp.campana != camp]
    m_gc = otros.groupby(['grupo', 'cultivo'])[OBJ].mean()
    m_c = otros.groupby(['cultivo', 'campana'])[OBJ].mean().groupby('cultivo').mean()
    s = (meta.campana.values == camp) & (meta.clase.values < 3)
    claves = list(zip(meta.grupo.values[s], meta.cultivo.values[s]))
    hist_depto[s] = pd.Series(claves).map(m_gc).values.astype(float)
    hist_zona[s] = meta.cultivo[s].map(m_c).values.astype(float)
lotes['rinde_hist_depto'] = hist_depto.round(0)
lotes['rinde_hist_cultivo'] = hist_zona.round(0)
if relleno is not None:
    lotes['pct_interpolado'] = (100 * relleno.mean(axis=1)).round(1)
lotes['tiene_serie'] = False   # se completa en la sección 6
print(lotes.head(3).T.to_string())

                                             0                       1              2
id                                           0                       1              2
lon                                  -62.09546              -61.968528     -62.765424
lat                                  -33.53658              -33.883958     -33.306701
campana                                2019_20                 2019_20        2019_20
provincia                             SANTA FE                SANTA FE        CORDOBA
departamento                     GENERAL LOPEZ           GENERAL LOPEZ          UNION
grupo                   SANTA FE|GENERAL LOPEZ  SANTA FE|GENERAL LOPEZ  CORDOBA|UNION
clase_inta                                Maíz                    Maíz           Maíz
clase_inta_n                                 0                       0              0
rinde_real                         9474.979691             9474.979691    7992.958608
pred_enero                                Maíz        

In [9]:
# Curvas de NDVI de cada lote y curvas típicas (media y desvío) por campaña, departamento y clase del INTA
np.savez_compressed(f'{LOCAL}/curvas_ndvi.npz', id=ids, ndvi=ndvi.astype(np.float16))
col_t = [f't{t:02d}' for t in range(T_FULL)]
df = pd.DataFrame(ndvi, columns=col_t)
df['campana'], df['grupo'], df['clase_inta'] = meta.campana.values, meta.grupo.values, meta.cultivo.values
tip = []
for claves, nivel in [(['campana', 'grupo', 'clase_inta'], 'departamento'), (['campana', 'clase_inta'], 'zona')]:
    g = df.groupby(claves)[col_t]
    media, desvio, n = g.mean(), g.std(), g.size()
    for clave in media.index:
        clave = clave if isinstance(clave, tuple) else (clave,)
        fila = dict(zip(claves, clave)); fila['nivel'] = nivel; fila['n_lotes'] = int(n.loc[clave])
        fila.update({f'media_{c}': round(float(v), 4) for c, v in zip(col_t, media.loc[clave])})
        fila.update({f'desvio_{c}': (None if np.isnan(v) else round(float(v), 4)) for c, v in zip(col_t, desvio.loc[clave])})
        tip.append(fila)
tipicas = pd.DataFrame(tip)
tipicas.loc[tipicas.nivel == 'zona', 'grupo'] = 'ZONA'
tipicas.to_csv(f'{LOCAL}/curvas_tipicas.csv', index=False)
print('Curvas típicas:', len(tipicas), '(departamento y zona)')

Curvas típicas: 874 (departamento y zona)


In [10]:
# Métricas para la página: por campaña y modo
y = meta.clase.values
met = {'clases': CLASES, 'aclaracion': 'Puntos de muestra en zonas puras (no un mapa completo ni una estimación de superficie). '
       'Cada campaña se predice con el modelo que no la vio al entrenar.', 'por_campana': {}}
for camp in campanas:
    s = meta.campana.values == camp
    met['por_campana'][camp] = {}
    for modo in ('enero', 'completa'):
        pred = lotes[f'pred_{modo}'].map({c: i for i, c in enumerate(CLASES)}).values
        # matriz "según el INTA (fila) -> según la red (columna)"
        mat = pd.crosstab(pd.Categorical(y[s], range(4)), pd.Categorical(pred[s], range(4)), dropna=False).values
        cultivo = s & (y < 3) & np.isfinite(lotes[f'rinde_{modo}'].values) & np.isfinite(lotes.rinde_real.values)
        # MAE por departamento: se promedia la predicción por departamento, cultivo y campaña
        g = lotes[cultivo].groupby(['grupo', 'clase_inta']).agg(real=('rinde_real', 'first'), pred=(f'rinde_{modo}', 'mean'),
                                                                historico=('rinde_hist_depto', 'first'))
        met['por_campana'][camp][modo] = {
            'f1_macro': round(float(f1_score(y[s], pred[s], average='macro', labels=range(4), zero_division=0)), 4),
            'lotes': int(s.sum()), 'aciertos': int((pred[s] == y[s]).sum()),
            'matriz_inta_vs_red': mat.tolist(),
            'mae_rinde_depto': round(float((g['real'] - g['pred']).abs().mean()), 0) if len(g) else None,
            'mae_promedio_depto': round(float((g['real'] - g['historico']).abs().mean()), 0) if len(g) and g['historico'].notna().any() else None,
        }
for modo in ('enero', 'completa'):
    f1s = [met['por_campana'][c][modo]['f1_macro'] for c in campanas]
    print(f'F1 macro {modo:8s}: ' + ' | '.join(f'{c} {v:.3f}' for c, v in zip(campanas, f1s)) + f' | media {np.mean(f1s):.3f}')
json.dump(met, open(f'{LOCAL}/metricas.json', 'w'), indent=1, ensure_ascii=False)

F1 macro enero   : 2019_20 0.947 | 2020_21 0.946 | 2021_22 0.933 | 2022_23 0.841 | 2023_24 0.921 | 2024_25 0.886 | media 0.912
F1 macro completa: 2019_20 0.949 | 2020_21 0.955 | 2021_22 0.945 | 2022_23 0.774 | 2023_24 0.951 | 2024_25 0.938 | media 0.919


## 6. Lotes de ejemplo con su serie cruda (para `/predict`)

In [11]:
rng = np.random.default_rng(SEED)
sel = []
for (camp, cl), idx in meta.groupby(['campana', 'clase']).indices.items():
    sel.extend(rng.choice(idx, size=min(N_EJEMPLOS_POR_GRUPO, len(idx)), replace=False))
sel = np.sort(np.array(sel))
np.savez_compressed(f'{LOCAL}/ejemplos/series.npz', id=ids[sel], campana=meta.campana.values[sel], bandas=X_u16[sel])
lotes.loc[sel, 'tiene_serie'] = True

# Un archivo descargable por campaña (un lote de cada clase), con el mismo formato que acepta /predict
for camp in campanas:
    for cl in range(len(CLASES)):
        i = sel[(meta.campana.values[sel] == camp) & (meta.clase.values[sel] == cl)]
        if len(i):
            i = i[0]
            np.savez_compressed(f'{LOCAL}/ejemplos/descargas/lote_{int(ids[i])}_{camp}.npz',
                                bandas=X_u16[i], campana=camp, id=int(ids[i]), bandas_orden=np.array(BANDAS_S2))
lotes.to_csv(f'{LOCAL}/lotes.csv', index=False)
json.dump({'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__, 'pandas': pd.__version__,
           'exportado': time.strftime('%Y-%m-%d %H:%M'), 'prueba': PRUEBA},
          open(f'{LOCAL}/versiones.json', 'w'), indent=1)
print(f'{len(sel)} lotes con serie cruda | {len(glob.glob(f"{LOCAL}/ejemplos/descargas/*.npz"))} archivos de ejemplo para descargar')

192 lotes con serie cruda | 24 archivos de ejemplo para descargar


## 7. Prueba final con `inferencia.py` y copia a Drive

In [12]:
# Un lote de ejemplo, como lo haría la API: con su campaña y sin campaña (promedio de los 6 modelos)
e = np.load(sorted(glob.glob(f'{LOCAL}/ejemplos/descargas/*.npz'))[0])
print('Con campaña:', json.dumps(M.predecir(e['bandas'], campana=str(e['campana'])), ensure_ascii=False, indent=1))
print('Sin campaña:', json.dumps(M.predecir(e['bandas']), ensure_ascii=False)[:300], '...')

tam = sum(os.path.getsize(p) for p in glob.glob(f'{LOCAL}/**/*', recursive=True) if os.path.isfile(p)) / 1e6
print(f'\ndatos_app: {tam:.0f} MB')
shutil.rmtree(f'{LOCAL}/__pycache__', ignore_errors=True)
shutil.rmtree(DESTINO, ignore_errors=True)
shutil.copytree(LOCAL, DESTINO, ignore=shutil.ignore_patterns('__pycache__'))
shutil.make_archive(DESTINO, 'zip', LOCAL)
print('Copiado a', DESTINO, 'y', DESTINO + '.zip')

Con campaña: {
 "campana_modelo": "2022_23",
 "enero": {
  "cultivo": "Soja 1ra",
  "probabilidades": {
   "Maíz": 0.001,
   "Soja 1ra": 0.9982,
   "Soja 2da": 0.0,
   "Otros": 0.0007
  },
  "rinde_kg_ha": 3302
 },
 "completa": {
  "cultivo": "Soja 1ra",
  "probabilidades": {
   "Maíz": 0.0047,
   "Soja 1ra": 0.9947,
   "Soja 2da": 0.0,
   "Otros": 0.0005
  },
  "rinde_kg_ha": 3135
 }
}
Sin campaña: {"campana_modelo": "promedio de los 6 modelos", "enero": {"cultivo": "Soja 1ra", "probabilidades": {"Maíz": 0.0365, "Soja 1ra": 0.9527, "Soja 2da": 0.0004, "Otros": 0.0104}, "rinde_kg_ha": 2276}, "completa": {"cultivo": "Soja 1ra", "probabilidades": {"Maíz": 0.0076, "Soja 1ra": 0.9907, "Soja 2da": 0 ...

datos_app: 26 MB
Copiado a /content/drive/MyDrive/TP_final_IA2/datos_app y /content/drive/MyDrive/TP_final_IA2/datos_app.zip
